# Reproducing SUPREME experiments

This notebook is a runnable companion to the README's **Reproducing the paper**
section and [`docs/reproducing_the_paper.md`](../docs/reproducing_the_paper.md).
It walks through installing the framework, running a quick smoke test, launching
the paper's experiment grid, rendering the result tables, and extending the
framework via the registry.

> **Hardware.** The unlearning pipeline needs a GPU (NVIDIA CUDA, or Apple
> Silicon MPS) for realistic runtimes. The paper's numbers were produced on a
> **single NVIDIA L40S (48 GB)**; multi-GPU runs are not bit-equivalent to
> single-GPU runs because of distributed gradient averaging, so reproduce the
> paper single-GPU. The shell cells below assume you run this notebook from a
> clone of the repository with the environment activated.

## 1. Install

Clone the repo and install the framework. For an exact reproduction of the
reported numbers, check out the tagged reference release **`v0.1.0-paper`**
first - later commits keep behaviour, defaults and seeds identical, but the tag
is the guaranteed reference point.

In [ ]:
# Run these in a shell (shown here for completeness):
# git clone https://github.com/pedroandreou/supreme-unlearning.git
# cd supreme-unlearning
# git checkout v0.1.0-paper      # exact paper reference (optional)

# Pick the requirements file matching your hardware, then install the package.
# NVIDIA (CUDA 12.1):
!pip install -r requirements.cuda_12_1.txt
# Apple Silicon (MPS) instead:
# !pip install -r requirements.mps.txt

!pip install -e .

## 2. Configure credentials

Evaluation results are logged **exclusively to Weights & Biases**, and ViT
weights are pulled from the Hugging Face Hub, so both tokens are required. Copy
the template and fill in your keys (see [`docs/environment_setup.md`](../docs/environment_setup.md)).

In [ ]:
# !cp .env.example .env
# then edit .env to set WANDB_API_KEY, the W&B username, and HF_TOKEN

from dotenv import load_dotenv
import os

load_dotenv()
print('WANDB_API_KEY set:', bool(os.getenv('WANDB_API_KEY')))
print('HF_TOKEN set:     ', bool(os.getenv('HF_TOKEN')))

## 3. Verify the install

`import supreme` is lightweight (it does not import the PyTorch/Lightning stack),
so it is a quick check that the package resolves and exposes its public API.

In [ ]:
import supreme

print('SUPREME version:', supreme.__version__)
print('Public API:', [n for n in supreme.__all__ if n.startswith(('register', 'run'))])

# Built-in components available out of the box:
print('Models   :', supreme.project_config.model_names)
print('Methods  :', supreme.project_config.all_methods)
print('Datasets :', supreme.project_config.dataset_names)
print('Metrics  :', supreme.project_config.evaluation_metrics)

## 4. Smoke test (one cell)

Before the full grid, confirm the end-to-end **train -> unlearn -> evaluate**
pipeline works on a single small cell. Re-running is safe: training checkpoints,
unlearning checkpoints and already-logged W&B results are detected and skipped.

Two equivalent ways to launch it:

In [ ]:
# (a) The experiment-grid driver (handles seeds, scenarios, the full pipeline).
#     macOS: use bash 4+, e.g. /opt/homebrew/bin/bash
!bash supreme/run_local.sh \
  --gpu 0 --models ViT --training-seeds 260 \
  --methods retrain,finetune,ssd \
  --strategies random_ --datasets Cifar10 \
  --forget-percs 0.01

In [ ]:
# (b) The Python API - same arguments as the supreme-unlearn console script.
#     (Run training first; see supreme-train -h / docs/script_arguments.md.)
# import supreme
# supreme.run_unlearning([
#     '-method', 'ssd', '-net', 'ViT', '-dataset', 'Cifar10',
#     '-type_of_unlearning_strategy', 'random_', '-seed', '260',
#     '-precision', '32-true', '-weight_path', '<path/to/trained.ckpt>',
# ])

## 5. Reproduce the paper's experiment grid

The command below matches the paper exactly: six unlearning methods plus the
retrain baseline, both scenarios (`fullclass`, `random_`), both architectures,
seeds 260-269, with a 0.1% forget set for the random-sample scenario. This is a
long-running production job - launch it on the GPU machine and let it populate
W&B. Interruptions are safe to resume (finished cells are skipped).

Pins Face Recognition requires a manual Kaggle download - see
[`docs/adding_pinsfacerecognition.md`](../docs/adding_pinsfacerecognition.md).

In [ ]:
!bash supreme/run_local.sh \
  --gpu 0 \
  --datasets PinsFaceRecognition \
  --models ResNet18,ViT \
  --strategies fullclass,random_ \
  --methods retrain,finetune,bad_teacher,random_labeling,unsir,ssd,lfssd \
  --forget-percs 0.001

## 6. Render the paper tables

Once the grid has populated W&B, export the metrics and render the three paper
LaTeX tables with
[`supreme/utils/wandb_utils/results_analysis/pins_paper_tables.ipynb`](../supreme/utils/wandb_utils/results_analysis/pins_paper_tables.ipynb).
The export-to-LaTeX workflow and troubleshooting notes are in
[`docs/reproducing_the_paper.md`](../docs/reproducing_the_paper.md) and
[`docs/tooling.md`](../docs/tooling.md).

## 7. Extending the framework (registry)

SUPREME is registry-based: you add datasets, models, unlearning methods,
baselines and metrics by **registering a module path** - from your own package,
with no edits to framework code. The runtime API below registers components for
the current process; installed plugin packages can register declaratively via
packaging entry points. Full interfaces and the Lightning Fabric integration
rules are in [`docs/extending.md`](../docs/extending.md).

In [ ]:
import supreme

# Register components living in *your* package (no edits to SUPREME):
supreme.register_unlearning_method('mymethod', 'my_pkg.mymethod')
supreme.register_model('MyNet', 'my_pkg.models:MyNet')
supreme.register_dataset(
    'MyDS', 'my_pkg.data:MyDS',
    root='/data/myds', class_dict={'cat': 0, 'dog': 1},
    rn_epochs=100, rn_milestones=[30, 60, 80],
)

# Registration keeps the framework's bookkeeping in sync, so the new names are
# immediately accepted by the CLI / validation:
print('mymethod registered:', 'mymethod' in supreme.project_config.all_methods)
print('MyDS registered    :', 'MyDS' in supreme.project_config.dataset_names)

# ...then run as usual:
# supreme.run_unlearning(['-method', 'mymethod', '-net', 'MyNet',
#                         '-dataset', 'MyDS', '-seed', '260', ...])